# 02. Cleaning the Hotline data

The raw Hotline table contains 55 jurisdictions. For the main geographic analysis, I'm focusing on the 50 states plus Washington, DC so that the data can be joined consistently with Census population and demographic data.

I am not deleting the remaining jurisdictions. They will be saved separately so the filtering decision stays transparent.

This notebook also adds state abbreviations and FIPS codes, which will make later joins with Census and mapping data easier.

In [1]:
import pandas as pd
from pathlib import Path


project_dir = Path.cwd()

if project_dir.name == "notebooks":
    project_dir = project_dir.parent


raw_file = (
    project_dir
    / "data"
    / "raw"
    / "hotline"
    / "national_hotline_state_statistics_2024_raw.csv"
)

clean_dir = project_dir / "data" / "cleaned"
clean_dir.mkdir(parents=True, exist_ok=True)


hotline = pd.read_csv(raw_file)

hotline.head()

,State,Signals received,% of total signals,Cases identified,% of total cases
0,Alabama,231,0.71%,105,0.87%
1,Alaska,46,0.14%,22,0.18%
2,Arizona,550,1.70%,300,2.50%
3,Arkansas,170,0.53%,80,0.67%
4,California,3378,10.46%,1734,14.45%


## Making the table easier to work with

The source table uses presentation-style column names and percentage symbols. I'm standardizing those fields here while leaving the original CSV untouched.

In [2]:
hotline.columns = [
    "state",
    "signals_received",
    "pct_total_signals",
    "cases_identified",
    "pct_total_cases"
]


hotline["state"] = (
    hotline["state"]
    .astype("string")
    .str.strip()
)


count_columns = [
    "signals_received",
    "cases_identified"
]

for column in count_columns:
    hotline[column] = pd.to_numeric(
        hotline[column]
        .astype(str)
        .str.replace(",", "", regex=False),
        errors="coerce"
    )


percent_columns = [
    "pct_total_signals",
    "pct_total_cases"
]

for column in percent_columns:
    hotline[column] = pd.to_numeric(
        hotline[column]
        .astype(str)
        .str.replace("%", "", regex=False),
        errors="coerce"
    )


hotline["year"] = 2024

hotline.head()

,state,signals_received,pct_total_signals,cases_identified,pct_total_cases,year
0,Alabama,231,0.71,105,0.87,2024
1,Alaska,46,0.14,22,0.18,2024
2,Arizona,550,1.70,300,2.50,2024
3,Arkansas,170,0.53,80,0.67,2024
4,California,3378,10.46,1734,14.45,2024


## Adding geographic identifiers

State names are useful for reading the data, but abbreviations and FIPS codes are more reliable when joining datasets.

I created a small reference table for the 50 states and Washington, DC and will use it to determine which Hotline rows belong in the main analysis.

In [3]:
state_reference = [
    ("Alabama", "AL", "01"),
    ("Alaska", "AK", "02"),
    ("Arizona", "AZ", "04"),
    ("Arkansas", "AR", "05"),
    ("California", "CA", "06"),
    ("Colorado", "CO", "08"),
    ("Connecticut", "CT", "09"),
    ("Delaware", "DE", "10"),
    ("District of Columbia", "DC", "11"),
    ("Florida", "FL", "12"),
    ("Georgia", "GA", "13"),
    ("Hawaii", "HI", "15"),
    ("Idaho", "ID", "16"),
    ("Illinois", "IL", "17"),
    ("Indiana", "IN", "18"),
    ("Iowa", "IA", "19"),
    ("Kansas", "KS", "20"),
    ("Kentucky", "KY", "21"),
    ("Louisiana", "LA", "22"),
    ("Maine", "ME", "23"),
    ("Maryland", "MD", "24"),
    ("Massachusetts", "MA", "25"),
    ("Michigan", "MI", "26"),
    ("Minnesota", "MN", "27"),
    ("Mississippi", "MS", "28"),
    ("Missouri", "MO", "29"),
    ("Montana", "MT", "30"),
    ("Nebraska", "NE", "31"),
    ("Nevada", "NV", "32"),
    ("New Hampshire", "NH", "33"),
    ("New Jersey", "NJ", "34"),
    ("New Mexico", "NM", "35"),
    ("New York", "NY", "36"),
    ("North Carolina", "NC", "37"),
    ("North Dakota", "ND", "38"),
    ("Ohio", "OH", "39"),
    ("Oklahoma", "OK", "40"),
    ("Oregon", "OR", "41"),
    ("Pennsylvania", "PA", "42"),
    ("Rhode Island", "RI", "44"),
    ("South Carolina", "SC", "45"),
    ("South Dakota", "SD", "46"),
    ("Tennessee", "TN", "47"),
    ("Texas", "TX", "48"),
    ("Utah", "UT", "49"),
    ("Vermont", "VT", "50"),
    ("Virginia", "VA", "51"),
    ("Washington", "WA", "53"),
    ("West Virginia", "WV", "54"),
    ("Wisconsin", "WI", "55"),
    ("Wyoming", "WY", "56")
]


state_lookup = pd.DataFrame(
    state_reference,
    columns=[
        "state",
        "state_abbr",
        "state_fips"
    ]
)

state_lookup.head()

,state,state_abbr,state_fips
0,Alabama,AL,01
1,Alaska,AK,02
2,Arizona,AZ,04
3,Arkansas,AR,05
4,California,CA,06


## Separating the main analysis area

A left join lets me keep every Hotline jurisdiction first and then see which ones do not match the 50-state + DC reference table.

This is preferable to silently dropping rows because I can inspect exactly what is being excluded.

In [4]:
hotline_geo = hotline.merge(
    state_lookup,
    on="state",
    how="left",
    validate="one_to_one"
)


other_jurisdictions = (
    hotline_geo[
        hotline_geo["state_fips"].isna()
    ]
    .copy()
    .reset_index(drop=True)
)


hotline_states = (
    hotline_geo[
        hotline_geo["state_fips"].notna()
    ]
    .copy()
    .reset_index(drop=True)
)


print("All Hotline jurisdictions:", len(hotline_geo))
print("States + DC:", len(hotline_states))
print("Other jurisdictions:", len(other_jurisdictions))

All Hotline jurisdictions: 55
States + DC: 51
Other jurisdictions: 4


In [5]:
other_jurisdictions[
    [
        "state",
        "signals_received",
        "cases_identified"
    ]
]

,state,signals_received,cases_identified
0,Guam,0,0
1,Northern Marianas Islands,0,0
2,Puerto Rico,8,10
3,U.S. Virgin Islands,0,0


## Quality checks

Before saving the cleaned dataset, I'm checking a few things that would cause problems later: duplicate states, missing counts, failed geographic matches, and unexpected row counts.

For the main analysis I expect exactly 51 rows: 50 states and Washington, DC.

In [6]:
checks = pd.DataFrame({
    "check": [
        "rows in main dataset",
        "unique states",
        "duplicate states",
        "missing signal counts",
        "missing case counts",
        "missing abbreviations",
        "missing FIPS codes"
    ],
    "result": [
        len(hotline_states),
        hotline_states["state"].nunique(),
        hotline_states["state"].duplicated().sum(),
        hotline_states["signals_received"].isna().sum(),
        hotline_states["cases_identified"].isna().sum(),
        hotline_states["state_abbr"].isna().sum(),
        hotline_states["state_fips"].isna().sum()
    ]
})

checks

,check,result
0,rows in main dataset,51
1,unique states,51
2,duplicate states,0
3,missing signal counts,0
4,missing case counts,0
5,missing abbreviations,0
6,missing FIPS codes,0


## A first derived measure

I am keeping the original counts, but I also want a simple measure of how many Hotline signals were received for each identified trafficking case.

This is descriptive only. It should not be interpreted as a probability that a signal becomes a trafficking case because a single situation can involve multiple contacts with the Hotline.

In [7]:
hotline_states["signals_per_case"] = (
    hotline_states["signals_received"]
    / hotline_states["cases_identified"]
).round(2)


hotline_states[
    [
        "state",
        "signals_received",
        "cases_identified",
        "signals_per_case"
    ]
].head(10)

,state,signals_received,cases_identified,signals_per_case
0,Alabama,231,105,2.20
1,Alaska,46,22,2.09
2,Arizona,550,300,1.83
3,Arkansas,170,80,2.12
4,California,3378,1734,1.95
5,Colorado,465,185,2.51
6,Connecticut,167,96,1.74
7,Delaware,51,32,1.59
8,District of Columbia,135,55,2.45
9,Florida,1830,832,2.20


## Raw rankings

I'm keeping a raw case ranking because it will be useful later when comparing it with a population-adjusted ranking.

I expect those rankings to differ. A state with a large population can rank highly in total reports without necessarily having a high reporting rate per resident.

In [8]:
hotline_states["raw_case_rank"] = (
    hotline_states["cases_identified"]
    .rank(
        method="min",
        ascending=False
    )
    .astype(int)
)


raw_ranking = (
    hotline_states[
        [
            "state",
            "state_abbr",
            "signals_received",
            "cases_identified",
            "raw_case_rank"
        ]
    ]
    .sort_values("raw_case_rank")
    .head(10)
    .reset_index(drop=True)
)

raw_ranking

,state,state_abbr,signals_received,cases_identified,raw_case_rank
0,California,CA,3378,1734,1
1,Texas,TX,2418,1360,2
2,Florida,FL,1830,832,3
3,New York,NY,1191,570,4
4,Illinois,IL,792,385,5
5,Georgia,GA,876,342,6
6,Michigan,MI,764,340,7
7,Ohio,OH,671,334,8
8,North Carolina,NC,638,301,9
9,Arizona,AZ,550,300,10


## Saving the cleaned Hotline data

I'm saving two files:

- the 51-jurisdiction dataset used for the main analysis
- the additional Hotline jurisdictions that are outside that scope

Keeping both makes the filtering decision reproducible instead of permanently removing information from the project.

In [9]:
main_file = (
    clean_dir
    / "hotline_2024_states_dc.csv"
)

other_file = (
    clean_dir
    / "hotline_2024_other_jurisdictions.csv"
)


hotline_states.to_csv(
    main_file,
    index=False
)

other_jurisdictions.to_csv(
    other_file,
    index=False
)


print("Main analysis file:")
print(main_file)

print("\nOther jurisdictions:")
print(other_file)

Main analysis file:
c:\Users\skybo\OneDrive\Documents\human-trafficking-resource-gap-analysis\data\cleaned\hotline_2024_states_dc.csv

Other jurisdictions:
c:\Users\skybo\OneDrive\Documents\human-trafficking-resource-gap-analysis\data\cleaned\hotline_2024_other_jurisdictions.csv
